In [ ]:
import pandas as pd
import numpy as np

# Config flags
auto_verbose = False 
show_plots = False   

train_path = "./data/split/train/train.csv"
test_path  = "./data/split/test/test.csv"

# Load raw data for UI and visualization
df_train = pd.read_csv(train_path)
df_test  = pd.read_csv(test_path)

# Try load saved address->cluster mapping from model6
import os
map_path = os.path.join(".", "model", "model6", "address_cluster_map.csv")
if os.path.isfile(map_path):
    address_cluster_map = pd.read_csv(map_path)
    if auto_verbose:
        print("Loaded saved address->cluster mapping from", map_path)
else:
    # Fallback: derive mapping from train if not found
    address_cluster_map = (
        df_train.groupby('address')['cluster']
          .agg(lambda s: s.mode().iat[0] if not s.mode().empty else -1)
          .reset_index()
          .rename(columns={'cluster':'cluster_mode'})
    )
    if auto_verbose:
        print("Fallback mapping computed from train data")

# Merge mapping to have cluster per row for visualization
if 'cluster_mode' in address_cluster_map.columns:
    addr_to_cluster = dict(zip(address_cluster_map['address'], address_cluster_map['cluster_mode']))
    df_train['cluster'] = df_train['address'].map(addr_to_cluster)
else:
    # if column named differently (edge cases)
    first_cluster_col = [c for c in address_cluster_map.columns if c != 'address'][0]
    addr_to_cluster = dict(zip(address_cluster_map['address'], address_cluster_map[first_cluster_col]))
    df_train['cluster'] = df_train['address'].map(addr_to_cluster)

if auto_verbose:
    print("Số lượng mỗi cluster (train):")
    print(df_train["cluster"].value_counts())

In [ ]:
# Mapping địa chỉ -> cluster (mode) (ưu tiên dùng file đã lưu)
import os
map_path = os.path.join(".", "model", "model6", "address_cluster_map.csv")
if os.path.isfile(map_path):
    address_cluster_map = pd.read_csv(map_path)
    if auto_verbose:
        print('\n=== Đọc mapping từ file ===')
        print(address_cluster_map.head())
else:
    address_cluster_map = (
        df_train.groupby('address')['cluster']
          .agg(lambda s: s.mode().iat[0] if not s.mode().empty else -1)
          .reset_index()
          .rename(columns={'cluster':'cluster_mode'})
    )
    if auto_verbose:
        print('\n=== Tự tính mapping từ train ===')
        print(address_cluster_map)

In [60]:
# Drop cột 'address' và (nếu có) 'region'; sau đó scale 'area'
from sklearn.preprocessing import StandardScaler
for df in (df_train, df_test):
    drop_cols = [c for c in ['address','region'] if c in df.columns]
    if drop_cols:
        df.drop(columns=drop_cols, inplace=True)

scaler_area = StandardScaler()
#df_train['area'] = scaler_area.fit_transform(df_train[['area']])
#df_test['area']  = scaler_area.transform(df_test[['area']])

In [61]:
import torch

def build_dataset(df):
    feature_cols = ["area", "bedrooms"]
    X = torch.tensor(df[feature_cols].values, dtype=torch.float32)
    y = torch.tensor(df[["price"]].values, dtype=torch.float32)
    return X, y

X0_train, y0_train = build_dataset(cluster_0_train)
X1_train, y1_train = build_dataset(cluster_1_train)
X2_train, y2_train = build_dataset(cluster_2_train)

X0_test, y0_test = build_dataset(cluster_0_test)
X1_test, y1_test = build_dataset(cluster_1_test)
X2_test, y2_test = build_dataset(cluster_2_test)

In [ ]:
# Load pre-trained weights from model/model6 instead of training
import os
import torch.nn as nn

models = {}

weights_dir = os.path.join(".", "model", "model6")
expected = [0, 1, 2]
missing = []

for idx in expected:
    weight_file = os.path.join(weights_dir, f"cluster_{idx}.pt")
    if not os.path.isfile(weight_file):
        missing.append(weight_file)

if missing:
    print("Không tìm thấy trọng số cho các cluster sau:")
    for f in missing:
        print(" -", f)
    print("Vui lòng chạy notebook 'notebook/model6.ipynb' để train và lưu trọng số trước.")
else:
    # Create model per cluster (2 input features: area, bedrooms) and load weights
    for idx in expected:
        m = nn.Linear(2, 1)
        state = torch.load(os.path.join(weights_dir, f"cluster_{idx}.pt"), map_location="cpu")
        m.load_state_dict(state)
        m.eval()
        models[idx] = m
    if auto_verbose:
        print("Đã load trọng số từ:", weights_dir)

In [64]:
# Notebook-based progress display
from IPython.display import display, clear_output
import time

def train_cluster_model_verbose(X_train, y_train, epochs=10000, lr=1e-3, log_every=100, title=""):
    model = nn.Linear(X_train.shape[1], 1)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    for ep in range(1, epochs+1):
        pred = model(X_train)
        loss = criterion(pred, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if ep % log_every == 0 or ep in (1, epochs):
            clear_output(wait=True)
            display(f"{title} Epoch {ep}/{epochs} - Loss: {loss.item():.6f}")
            time.sleep(0.01)
    return model


In [65]:
from ipywidgets import VBox, HBox, FloatText, IntText, Dropdown, Button, Output, Layout
from IPython.display import display, clear_output
import numpy as np
import torch
import matplotlib.pyplot as plt

# Danh sách địa chỉ dựa trên bảng mapping (tạo ở cell trước)
if 'address_cluster_map' in globals():
    address_list = sorted(address_cluster_map['address'].dropna().unique().tolist())
else:
    # Fallback nếu chưa có mapping (ít gặp): lấy từ df_train nếu còn cột address
    address_list = sorted(df_train['address'].dropna().unique().tolist()) if 'address' in df_train.columns else []

# Tạo dict map địa chỉ -> cluster_mode
addr_to_cluster = {}
if 'address_cluster_map' in globals():
    addr_to_cluster = dict(zip(address_cluster_map['address'], address_cluster_map['cluster_mode']))

# Widgets (xếp dọc thay vì ngang)
address_w = Dropdown(options=address_list, description='Address:', layout=Layout(width='400px'))
area_w    = FloatText(value=float(df_train['area'].median()) if 'area' in df_train.columns else 50.0, description='Area:', layout=Layout(width='300px'))
bed_w     = IntText(value=int(df_train['bedrooms'].median()) if 'bedrooms' in df_train.columns else 2, description='Bedrooms:', layout=Layout(width='300px'))
run_btn   = Button(description='Predict', button_style='primary', layout=Layout(width='150px'))
out       = Output()

# Handler
@run_btn.on_click
def _on_run(_):
    with out:
        clear_output(wait=True)
        if 'models' not in globals() or not models:
            print('Models chưa sẵn sàng. Hãy chạy toàn bộ notebook trước.')
            return
        sel_addr = address_w.value
        cluster_id = addr_to_cluster.get(sel_addr, 1)
        if cluster_id not in models:
            if 1 in models:
                cluster_id = 1
            else:
                cluster_id = sorted(models.keys())[0]
        
        x_np = np.array([[area_w.value, float(bed_w.value)]], dtype=np.float32)
        x_t = torch.tensor(x_np, dtype=torch.float32)
        with torch.no_grad():
            yhat = models[cluster_id](x_t).cpu().numpy().ravel()[0]
        print(f"Address: {sel_addr} -> Cluster {cluster_id}")
        print(f"Predicted price = {yhat:,.2f}")
        
        # Visualization
        # Get cluster data
        cluster_data = df_train[df_train['cluster'] == cluster_id].copy()
        if len(cluster_data) == 0:
            print("Không có dữ liệu cho cluster này.")
            return
            
        # Plot blue points (cluster data)
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.scatter(cluster_data['area'], cluster_data['price'], 
                  c='blue', alpha=0.5, label='Cluster data')
        
        # Red trend line (model prediction)
        area_range = np.linspace(cluster_data['area'].min(), 
                                cluster_data['area'].max(), 100)
        X_line = np.column_stack([
            area_range,
            np.full(100, bed_w.value)
        ]).astype(np.float32)
        X_line_t = torch.tensor(X_line, dtype=torch.float32)
        with torch.no_grad():
            y_line = models[cluster_id](X_line_t).cpu().numpy().ravel()
        ax.plot(area_range, y_line, 'r-', linewidth=2, label='Model trend')
        
        # Yellow point (user input)
        ax.scatter([area_w.value], [yhat], c='yellow', s=200, 
                  edgecolors='black', linewidths=2, 
                  label='Your input', zorder=5)
        
        ax.set_xlabel('Area (m²)')
        ax.set_ylabel('Price')
        ax.set_title(f'Cluster {cluster_id} - Price vs Area')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

# UI dọc
ui = VBox([
    address_w,
    area_w,
    bed_w,
    run_btn,
    out
])
display(ui)